# Feature Calculation for Model Input

## Preparations

Install requirements

In [ ]:
%pip install -r ../requirements.txt

Import libraries

In [5]:
from pathlib import Path
import multiprocessing

import numpy as np
import pandas as pd
import polars as pl
import sympy as sp
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from joblib import Parallel, delayed

Initialize functions & Set configurations

In [40]:
def hampel(y, window_size, simg=3):
    new_y = y.copy()
    for i in range(window_size, len(y) - window_size):
        window = y[(i - window_size):(i + window_size)]
        med = np.median(window)
        mad = np.median(np.abs(window - med))
        if np.abs(y[i] - med) > simg * mad:
            new_y[i] = med
    return new_y


def add_indices(df):
    return df.with_columns([
        ((pl.col("green") - pl.col("blue")) / (pl.col("green") + pl.col("blue"))).alias("ndyi"),
        ((pl.col("nir") - pl.col("swir1")) / (pl.col("nir") + pl.col("swir1"))).alias("ndmi"),
        ((pl.col("nir") * 0.1 - pl.col("red")) / (pl.col("nir") * 0.1 + pl.col("red"))).alias("wrdvi"),
    ])

def prepare_data(path, var_group):
    df = pl.read_csv(path)
    cols = [c for c in df.columns if any(band in c for band in var_group)]

    df = df.select(["field_id"] + cols)
    df_long = df.unpivot(
        index=["field_id"],
        on=cols,
        variable_name="variable",
        value_name="value"
    )
    df_long = df_long.filter(pl.col("value").is_not_null())

    df_long = df_long.with_columns([
        pl.col("variable").str.split("_").list.get(-2).alias("date"),
        pl.col("variable").str.split("_").list.get(-1).alias("band"),
    ])

    df_long = df_long.with_columns(
        pl.col("date").str.strptime(pl.Date, "%Y%m%d")
    )

    result_df = df_long.pivot(
        values="value",
        index=["field_id", "date"],
        on="band",
        aggregate_function="first"
    )
    result_df = result_df.with_columns([
        pl.col(var_group).cast(pl.Float64)
    ])

    result_df = result_df.with_columns([
        pl.col("date").dt.ordinal_day().alias("DOY"),
        pl.col("date").dt.month().alias("month"),
    ])
    

    return result_df.sort(["field_id", "DOY"])


def process_spectral_features_means(group_df, index, precomputed_values=None):
    group_df = group_df.filter(pl.col("DOY") >= 90)
    if len(group_df) < 2:
        return None

    x = group_df["DOY"].to_numpy().astype(float)
    y = precomputed_values if precomputed_values is not None else hampel(group_df[index].to_numpy().astype(float), 3)

    f = interp1d(x, y, kind='linear', fill_value='extrapolate')
    y_interp = f(X_VALUES)

    result = {'field_id': group_df['field_id'][0]}
    result[f'{index}_min'] = float(np.min(y_interp))
    result[f'{index}_max'] = float(np.max(y_interp))
    result[f'{index}_doy_min'] = float(X_VALUES[np.argmin(y_interp)])
    result[f'{index}_doy_max'] = float(X_VALUES[np.argmax(y_interp)])

    synth = pl.DataFrame({'DOY': X_VALUES, index: y_interp})
    synth = synth.with_columns(
        pl.col("DOY").map_elements(
            lambda x: pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(x) - 1),
            return_dtype=pl.Datetime
        ).dt.month().alias("month")
    )

    monthly = synth.group_by("month").agg(pl.col(index).median().alias("median"))
    for month in range(4, 11):
        row = monthly.filter(pl.col("month") == month)
        result[f'median_{index}_fitted_{month}'] = row['median'][0] if len(row) else None

    return result


def double_logistic_function(t, wNDVI, mNDVI, S, A, mS, mA):
    sigmoid1 = 1 / (1 + np.exp(-mS * (t - S)))
    sigmoid2 = 1 / (1 + np.exp(mA * (t - A)))
    seasonal_term = sigmoid1 + sigmoid2 - 1
    return wNDVI + (mNDVI - wNDVI) * seasonal_term


def fit_curve(t, ndvi_observed, bounds):
    initial_guess = [np.min(ndvi_observed), np.max(ndvi_observed), 0, 365, 0.1, 0.1]
    try:
        params, _ = curve_fit(
            double_logistic_function,
            t,
            ndvi_observed,
            p0=initial_guess,
            bounds=bounds,
            maxfev=5000,
        )
        return params
    except Exception:
        return None

def sort_extrema_points(extrema_points_x, doy_max):
    """Sort extremum points based on comparison with doy_max."""
    less_than_doy_max = [x for x in extrema_points_x if x < doy_max]
    greater_than_doy_max = [x for x in extrema_points_x if x > doy_max]
    
    return {
        'start_of_growth': min(less_than_doy_max) if less_than_doy_max else None,
        'end_of_growth': max(less_than_doy_max) if less_than_doy_max else None,
        'start_of_decay': min(greater_than_doy_max) if greater_than_doy_max else None,
        'end_of_decay': max(greater_than_doy_max) if greater_than_doy_max else None
    }


def process_spectral_features_curve(group_df, index, bounds, precomputed_values=None):
    group_df = group_df.filter(pl.col("DOY") >= 90)
    if len(group_df) < 6:
        return None

    x = group_df["DOY"].to_numpy().astype(float)
    y = precomputed_values if precomputed_values is not None else hampel(group_df[index].to_numpy().astype(float), 3)

    params = fit_curve(x, y, bounds)
    if params is None or np.isnan(params).any():
        return None

    wNDVI, mNDVI, S, A, mS, mA = params
        
    finer_values = double_logistic_function(X_VALUES, *params)
    doy_max=float(X_VALUES[np.argmax(finer_values)])
    

    f_quadruple_prime_lambdified = sp.lambdify(symbols_for_lambdify, fourth_derivative_sym, 'numpy')
    fourth_derivative_values = f_quadruple_prime_lambdified(X_VALUES, *params)
    zero_crossings_fourth = np.where(np.diff(np.sign(fourth_derivative_values)))[0]
    extrema_points_third_x = X_VALUES[zero_crossings_fourth]
    extrema_points_third_x = sorted(set(extrema_points_third_x))
    
    sorted_points = sort_extrema_points(extrema_points_third_x, doy_max)
    
    result = {
        'field_id': group_df['field_id'][0],
        f'{index}_wNDVI': params[0],
        f'{index}_mNDVI': params[1],
        f'{index}_S': params[2],
        f'{index}_A': params[3],
        f'{index}_mS': params[4],
        f'{index}_mA': params[5],
        f'{index}_doy_max': doy_max,
        f'{index}_max': float(np.max(finer_values)),
        f'{index}_start_of_growth': sorted_points.get('start_of_growth'),
        f'{index}_end_of_growth': sorted_points.get('end_of_growth'),
        f'{index}_start_of_decay': sorted_points.get('start_of_decay'),
        f'{index}_end_of_decay': sorted_points.get('end_of_decay'),
        f'{index}_max_growth': double_logistic_function(sorted_points['end_of_growth'], *params) if sorted_points['end_of_growth'] else None,
        f'{index}_mean_growth': double_logistic_function(S, *params),
        f'{index}_min_growth': double_logistic_function(sorted_points['start_of_growth'], *params) if sorted_points['start_of_growth'] else None,
        f'{index}_max_decay': double_logistic_function(sorted_points['start_of_decay'], *params) if sorted_points['start_of_decay'] else None,
        f'{index}_min_decay': double_logistic_function(sorted_points['end_of_decay'], *params) if sorted_points['end_of_decay'] else None,
        f'{index}_mean_decay': double_logistic_function(A, *params),
    }

    synth = pl.DataFrame({'DOY': X_VALUES, index: finer_values})
    synth = synth.with_columns(
        pl.col("DOY").map_elements(
            lambda x: pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(x) - 1),
            return_dtype=pl.Datetime,
        ).dt.month().alias("month")
    )
    monthly = synth.group_by("month").agg(pl.col(index).median().alias("median"))
    for month in range(4, 11):
        row = monthly.filter(pl.col("month") == month)
        result[f'median_{index}_fitted_{month}'] = row['median'][0] if len(row) else None

    return result

def process_ndyi_features(group_df, index, precomputed_values=None):
    group_df = group_df.filter(pl.col("DOY") >= 90)
    if len(group_df) == 0:
        return None
    x = group_df["DOY"].to_numpy().astype(float)
    y = precomputed_values if precomputed_values is not None else hampel(group_df[index].to_numpy().astype(float), 3)

    result = {'field_id': group_df['field_id'][0]}
    result[f'{index}_min'] = float(np.min(y))
    result[f'{index}_max'] = float(np.max(y))
    result[f'{index}_doy_min'] = float(x[np.argmin(y)])
    result[f'{index}_doy_max'] = float(x[np.argmax(y)])

    return result


def calculate_meteo_features(file: str) -> pl.DataFrame:
    df = prepare_data(file, ['temperature', 'precipitation'])

    condition = df["temperature"] > 10

    com_calculated = (
        df.group_by(["field_id", "month"])
        .agg([
            pl.col("temperature").median().alias("median_t"),
            pl.col("precipitation").median().alias("median_prec"),
            pl.col("precipitation").sum().alias("sum_prec")
        ])
    )

    temp_calculated = (
        df.filter(condition)
        .group_by(["field_id", "month"])
        .agg([
            pl.col("temperature").sum().alias("sum_t")
        ])
    )

    calculated_df = com_calculated.join(
        temp_calculated,
        on=["field_id", "month"],
        how="left"
    )

    result_df = None

    for col in ["median_t", "sum_t", "median_prec", "sum_prec"]:
        pivot_df = calculated_df.pivot(
            index=["field_id"],
            on="month",
            values=col,
            aggregate_function=None
        )

        pivot_df = pivot_df.rename({
            str(m): f"{col}_{m}"
            for m in calculated_df["month"].unique().to_list()
        })

        if result_df is None:
            result_df = pivot_df
        else:
            result_df = result_df.join(pivot_df, on="field_id", how="left")

    return result_df


def run_pipeline(pipeline_name, index, grouped_data, bounds_config=None):
    func = FEATURE_PIPELINES[pipeline_name]

    if pipeline_name == "curve":
        return Parallel(n_jobs=multiprocessing.cpu_count())(
            delayed(func)(group, index, bounds_config[index], cache.get(index))
            for group, cache in grouped_data
        )

    return Parallel(n_jobs=multiprocessing.cpu_count())(
        delayed(func)(group, index, cache.get(index))
        for group, cache in grouped_data
    )

def calculate_spectral_features(spectral_path):
    df = prepare_data(spectral_path, BASIC_INDICES)
    df = add_indices(df)

    grouped_data = []
    for _, group in df.group_by("field_id"):
        filtered_group = group.filter(pl.col("DOY") >= 90)
        if len(filtered_group) == 0:
            continue

        hampel_cache = {}
        for index in FEATURE_CONFIG.keys():
            values = filtered_group[index].to_numpy().astype(float)
            hampel_cache[index] = hampel(values, 3) if len(values) > 0 else None

        grouped_data.append((filtered_group, hampel_cache))

    all_field_ids = df["field_id"].unique().to_frame()
    all_features = []

    bounds_config = {
        'wrdvi': ([-1, -1, 0, 0, 0, 0], [1, 1, 365, 365, 1, 1]),
        'ndmi': ([-0.2, -0.2, 0, 0, 0, 0], [1, 1, 365, 365, 1, 1]),
    }

    for index, cfg in FEATURE_CONFIG.items():
        pipeline = cfg["pipeline"]

        print(f"Calculating {pipeline} feature {index}")

        results = run_pipeline(
            pipeline,
            index,
            grouped_data,
            bounds_config=bounds_config
        )

        results = [r for r in results if r is not None]
        if results:
            all_features.append(pl.DataFrame(results))

    merged = all_field_ids
    for feat_df in all_features:
        merged = merged.join(feat_df, on='field_id', how='left')

    return merged


def build_features(spectral_path, meteo_path, OUTPUT_FILE):
    print("Calculating spectral features...")
    spectral_data=calculate_spectral_features(spectral_path)

    print("Calculating meteorological features...")
    meteodata=calculate_meteo_features(meteo_path)
    
    print("Building features...")
    merged = spectral_data.join(meteodata, on='field_id', how='left')
    
    merged.write_parquet(OUTPUT_FILE)
    print(f"Saved features to {OUTPUT_FILE}")

BASIC_INDICES = ['red', 'nir', 'blue', 'swir1', 'green', 'swir2']
FEATURE_CONFIG = {
    "red":   {"pipeline": "mean"},
    "nir":   {"pipeline": "mean"},
    "blue":  {"pipeline": "mean"},
    "swir1": {"pipeline": "mean"},
    "green": {"pipeline": "mean"},
    "swir2": {"pipeline": "mean"},

    "ndyi":  {"pipeline": "extrema"},
    "ndmi":  {"pipeline": "curve"},
    "wrdvi": {"pipeline": "curve"},
}

FEATURE_PIPELINES = {
    "mean": process_spectral_features_means,
    "curve": process_spectral_features_curve,
    "extrema": process_ndyi_features,
}

X_VALUES = np.linspace(1, 365, 2000)

x = sp.symbols('x')
wNDVI_sym, mNDVI_sym, S_sym, A_sym, mS_sym, mA_sym = sp.symbols('wNDVI mNDVI S A mS mA')
sigmoid1_sym = 1 / (1 + sp.exp(-mS_sym * (x - S_sym)))
sigmoid2_sym = 1 / (1 + sp.exp(mA_sym * (x - A_sym)))
seasonal_term_sym = sigmoid1_sym + sigmoid2_sym - 1
sympy_dlf_template = wNDVI_sym + (mNDVI_sym - wNDVI_sym) * seasonal_term_sym
first_derivative_sym = sp.diff(sympy_dlf_template, x)
second_derivative_sym = sp.diff(first_derivative_sym, x)
third_derivative_sym = sp.diff(second_derivative_sym, x)
fourth_derivative_sym = sp.diff(third_derivative_sym, x)

symbols_for_lambdify = [x, wNDVI_sym, mNDVI_sym, S_sym, A_sym, mS_sym, mA_sym]

## Run Calculations

In [42]:
spectral_data_path = Path('../data/raw/fields_spectral_data.csv')
meteo_data_path ='../data/raw/fields_meteo_data.csv'
output_file = "../data/processed/input_data_for_model.parquet"

print(f'start processing {spectral_data_path.name}...')
try:
    build_features(spectral_data_path, meteo_data_path, output_file)
except Exception as e:
    print(f'error during processing {spectral_data_path.name} - {e}')

start processing fields_spectral_data.csv...
Calculating spectral features...
Calculating mean feature red
Calculating mean feature nir
Calculating mean feature blue
Calculating mean feature swir1
Calculating mean feature green
Calculating mean feature swir2
Calculating extrema feature ndyi
Calculating curve feature ndmi


<lambdifygenerated-1>:2: RuntimeWarning: overflow encountered in power
  return (mNDVI - wNDVI)*(-mA**4*exp(mA*(-A + x))/(exp(mA*(-A + x)) + 1)**2 + 14*mA**4*exp(2*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**3 - 36*mA**4*exp(3*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**4 + 24*mA**4*exp(4*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**5 - mS**4*exp(-mS*(-S + x))/(1 + exp(-mS*(-S + x)))**2 + 14*mS**4*exp(-2*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**3 - 36*mS**4*exp(-3*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**4 + 24*mS**4*exp(-4*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**5)
<lambdifygenerated-1>:2: RuntimeWarning: overflow encountered in exp
  return (mNDVI - wNDVI)*(-mA**4*exp(mA*(-A + x))/(exp(mA*(-A + x)) + 1)**2 + 14*mA**4*exp(2*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**3 - 36*mA**4*exp(3*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**4 + 24*mA**4*exp(4*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**5 - mS**4*exp(-mS*(-S + x))/(1 + exp(-mS*(-S + x)))**2 + 14*mS**4*exp(-2*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**3 - 36*mS**4*exp(-3*mS*(-S + x))/(1

Calculating curve feature wrdvi


<lambdifygenerated-2>:2: RuntimeWarning: overflow encountered in power
  return (mNDVI - wNDVI)*(-mA**4*exp(mA*(-A + x))/(exp(mA*(-A + x)) + 1)**2 + 14*mA**4*exp(2*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**3 - 36*mA**4*exp(3*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**4 + 24*mA**4*exp(4*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**5 - mS**4*exp(-mS*(-S + x))/(1 + exp(-mS*(-S + x)))**2 + 14*mS**4*exp(-2*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**3 - 36*mS**4*exp(-3*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**4 + 24*mS**4*exp(-4*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**5)
<lambdifygenerated-2>:2: RuntimeWarning: overflow encountered in power
  return (mNDVI - wNDVI)*(-mA**4*exp(mA*(-A + x))/(exp(mA*(-A + x)) + 1)**2 + 14*mA**4*exp(2*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**3 - 36*mA**4*exp(3*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**4 + 24*mA**4*exp(4*mA*(-A + x))/(exp(mA*(-A + x)) + 1)**5 - mS**4*exp(-mS*(-S + x))/(1 + exp(-mS*(-S + x)))**2 + 14*mS**4*exp(-2*mS*(-S + x))/(1 + exp(-mS*(-S + x)))**3 - 36*mS**4*exp(-3*mS*(-S + x))/

Calculating meteorological features...
Building features...
Saved features to ../data/processed/input_data_for_model.parquet
